> 📌 このノートブックは [Learn-Prompt-Hacking](https://github.com/TrustAI-laboratory/Learn-Prompt-Hacking) の日本語版です。

# 🔧 環境セットアップ — LLM API の準備

このノートブックでは、本コース全体で使用する LLM（大規模言語モデル）API の接続設定を行います。
本コースでは **local LLM** を推奨していますが、**OpenAI API**／**GitHub Models** も利用可能です。

## ⚠️ 最新 LLM のプロンプトインジェクション対策について

**最新の商用 LLM（GPT-4o, GPT-4o-mini, Claude 等）はサーバー側でプロンプトインジェクション対策が施されており**、教材のプロンプト例が意図どおりに動作しない（攻撃が拒否される）ケースがあります。

### なぜ攻撃が通らないのか？

現在の主要 LLM は、以下のような多層防御を実装しています:

- **Instruction Hierarchy（指示の優先順位付け）**: システムプロンプト ＞ ユーザー入力 の優先順位を訓練段階で組み込み、ユーザー入力による指示の上書きを防止（参考: [The Instruction Hierarchy — OpenAI, 2024](https://arxiv.org/abs/2404.13208)）
- **安全性アラインメント**: RLHF（人間のフィードバックによる強化学習）やルールベースのフィルタリングにより、有害な出力を抑制
- **サーバーサイドフィルタリング**: API ゲートウェイ段階でのコンテンツフィルタ（例: OpenAI の [Moderation API](https://platform.openai.com/docs/guides/moderation)）

そのため、本教材で学ぶ攻撃手法の一部は **「防御が機能している実例」** として観察することになります。

## パッケージのインストール

必要なパッケージをインストールします。

In [ ]:
%pip install -q openai

## ローカル LLM のセットアップ（Ollama）

[Ollama](https://ollama.com/) は、ローカル環境で LLM を簡単に実行できるツールです。OpenAI 互換の API サーバーを内蔵しており、本コースの `llm_client.py` からそのまま利用できます。

### 1. Ollama のインストール

| OS | コマンド |
|----|---------|
| **macOS** (Homebrew) | `brew install ollama` |
| **Linux** | `curl -fsSL https://ollama.com/install.sh \| sh` |
| **Windows** | [公式サイト](https://ollama.com/download) からダウンロード |

### 2. モデルのダウンロードと起動

```bash
# 軽量モデルをダウンロード（約 3GB）
ollama pull gemma3:4b

# Ollama サーバーを起動
ollama serve &
```

> 💡 **推奨モデル**: `gemma3:4b`（Google の軽量モデル、日本語対応、メモリ 8GB 以上推奨）
>
> その他の選択肢:
> - `llama3.2:3b` — Meta の軽量モデル（約 2GB）
> - `phi4-mini` — Microsoft の小型モデル（約 2.5GB）
> - `qwen3:4b` — Alibaba のモデル（日本語に強い、約 2.7GB）

### 3. 環境変数の設定

ローカル LLM を使うには、以下の環境変数を設定します:

```bash
export LLM_PROVIDER=local
# モデル名を変更する場合:
# export LLM_MODEL=llama3.2:3b
```

または、ノートブック内で直接設定:

In [ ]:
# ローカル LLM を使う場合はコメントを外して実行
# import os
# os.environ["LLM_PROVIDER"] = "local"
# os.environ["LLM_MODEL"] = "gemma3:4b"  # 使用するモデル名

> ⚠️ 環境変数を変更した場合は、**カーネルを再起動** してから `llm_client` を再読み込みしてください。
>
> ローカル LLM は `http://localhost:11434` で動作します。`ollama serve` が実行中であることを確認してください。

### 4. Ollama の停止方法

Ollama を停止するには、以下の方法があります:

**方法1: プロセスを見つけて停止（macOS/Linux）**
```bash
# Ollama のプロセスを確認
ps aux | grep "ollama serve" | grep -v grep

# プロセスを停止
pkill -f "ollama serve"
```

**方法2: Ctrl+C で停止**
`ollama serve` を実行したターミナルで `Ctrl+C` を押して停止できます。

**方法3: macOS の場合（バックグラウンドサービスとして実行している場合）**
```bash
# サービスを停止
brew services stop ollama
```

**動作確認**
```bash
# Ollama が停止したことを確認（何も表示されなければ停止済み）
ps aux | grep "ollama serve" | grep -v grep
```

> 💡 **「address already in use」エラーが出た場合**: 既に Ollama が起動しているので、新しく起動する必要はありません。停止したい場合のみ上記の方法を使用してください。

## 動作確認

実際にクライアントを読み込んで、LLM を呼び出してみましょう。

In [ ]:
import sys
sys.path.insert(0, "..")
from utils.llm_client import call_llm, call_llm_and_print, md_print, MODEL_NAME, PROVIDER

print(f"プロバイダ: {PROVIDER}")
print(f"モデル:     {MODEL_NAME}")
print("✅ クライアントの初期化に成功しました")

### テスト: 簡単な API 呼び出し

`call_llm()` は応答テキストを返し、`call_llm_and_print()` はプロンプトと応答を Markdown で表示します。

In [ ]:
# call_llm: 応答テキストのみを取得
response = call_llm("こんにちは！あなたは何のモデルですか？1文で答えてください。")
print(response)

In [ ]:
# call_llm_and_print: プロンプトと応答を Markdown で表示
call_llm_and_print("AIセキュリティとは何ですか？2文で簡潔に説明してください。")

### テスト: システムプロンプト付き呼び出し

`system` パラメータでシステムプロンプト（AI の役割設定）を指定できます。

In [ ]:
# システムプロンプトで役割を指定
call_llm_and_print(
    "プロンプトインジェクションとは？",
    system="あなたはサイバーセキュリティの専門家です。初心者にも分かりやすく説明してください。"
)

## 各ノートブックでの使い方

他のノートブックでは、以下の2行を冒頭に書くだけで LLM クライアントを利用できます:

```python
import sys; sys.path.insert(0, "..")
from utils.llm_client import call_llm, call_llm_and_print, md_print
```

---

✅ **セットアップ完了！** 次のノートブック [01_Introduction_to_AI.ipynb](01.😄Introduction_to_AI.ipynb) に進みましょう。

## GitHub Models とは

[GitHub Models](https://github.com/marketplace/models) は、GitHub が提供する AI モデル推論サービスです。

**特徴:**
- OpenAI SDK（`openai` パッケージ）と **完全互換** — `base_url` を差し替えるだけで利用可能
- GPT-4o, GPT-4o-mini, Phi-4, Llama, Mistral など複数のモデルに対応
- GitHub の Personal Access Token（PAT）または `gh auth token` で認証

### 利用プランと制限
[GitHub Models ドキュメント](https://docs.github.com/en/github-models/use-github-models/prototyping-with-ai-models#rate-limits) を確認してください。

教材用途（学習ペースでの API 呼び出し）であれば **Free プランでも十分** です。

## GitHub CLI のインストールと認証

GitHub Models を利用するには GitHub Token が必要です。**GitHub CLI を使う方法が最も簡単** です。

### 1. GitHub CLI のインストール

| OS | コマンド |
|----|---------|
| **macOS** (Homebrew) | `brew install gh` |
| **Windows** (winget) | `winget install --id GitHub.cli` |

> 📖 その他の OS・インストール方法: https://github.com/cli/cli#installation

---

### 2. GitHub CLI でログイン

```bash
gh auth login
```

対話形式で以下の質問に答えていきます:

```
? Where do you use GitHub?
> GitHub.com          ← これを選択

? What is your preferred protocol for Git operations on this host?
> SSH                 ← SSH 推奨（HTTPS でも可）

? Upload your SSH public key to your GitHub account?
> Skip                ← Skip でOK

? How would you like to authenticate GitHub CLI?
> Login with a web browser   ← ブラウザ認証を選択
```

ブラウザ認証を選択すると、**ワンタイムコード** が表示されます:

```
! First copy your one-time code: XXXX-XXXX
Press Enter to open https://github.com/login/device in your browser...
```

Enter を押すとブラウザが開くので、以下の手順で認証します:

**① 「Continue」をクリック:**

![Continue](../resources/images/cli-login_01_continue-check.png)

**② ワンタイムコードを入力:**

![ワンタイムコード入力](../resources/images/cli-login_02_input-one-time-pass.png)

**③ 権限を確認して「Authorize」:**

![Authorize](../resources/images/cli-login_03_permission-check.png)

**④ 認証完了！**

![認証完了](../resources/images/cli-login_04_cli-login-complete.png)

ターミナルに戻ると、ログイン完了のメッセージが表示されます:

```
✓ Authentication complete.
- gh config set -h github.com git_protocol ssh
✓ Configured git protocol
✓ Logged in as <あなたのユーザー名>
```

本コースの `utils/llm_client.py` は自動的に `gh auth token` を呼び出すので、**環境変数の設定は不要** です。

---

### 3. ログインの確認

以下のコマンドで正しくログインできているか確認できます:

```bash
gh auth status
```

---

### 補足: ログアウト

認証情報を削除したい場合は以下を実行します:

```bash
gh auth logout
```

---

### 代替方法: 環境変数で設定（CLI を使わない場合）

Personal Access Token（PAT）を作成して環境変数に設定する方法:

1. https://github.com/settings/tokens にアクセス
2. 「Generate new token (classic)」をクリック
3. スコープは特に不要（トークンの存在自体で認証される）
4. 生成されたトークンを環境変数に設定:

```bash
export GITHUB_TOKEN="ghp_xxxxxxxxxxxxxxxxxxxx"
```

### 代替方法: OpenAI API を使いたい場合

```bash
export LLM_PROVIDER="openai"
export OPENAI_API_KEY="sk-xxxxxxxxxxxxxxxxxxxx"
```

## utils/llm_client.py の仕組み

本コースでは、各ノートブックから共通の LLM クライアントを利用します。設定は `utils/llm_client.py` に集約されています。

### 主要なコンポーネント

| 名前 | 種別 | 説明 |
|------|------|------|
| `client` | OpenAI クライアント | API 呼び出し用のクライアントインスタンス |
| `call_llm(prompt)` | 関数 | プロンプトを送信し、応答テキストを返す |
| `call_llm_and_print(prompt)` | 関数 | プロンプトと応答を Markdown で表示（旧 `call_GPT` の代替） |
| `md_print(text)` | 関数 | テキストを Markdown として表示 |
| `MODEL_NAME` | 定数 | 使用中のモデル名（デフォルト: `openai/gpt-4o-mini`） |
| `PROVIDER` | 定数 | 使用中のプロバイダ（`"github"` / `"openai"` / `"local"`） |

### 環境変数による設定

| 環境変数 | デフォルト | 説明 |
|----------|-----------|------|
| `LLM_PROVIDER` | `"github"` | `"github"` / `"openai"` / `"local"` |
| `LLM_MODEL` | `"openai/gpt-4o-mini"` | 使用するモデル名 |
| `GITHUB_TOKEN` | (gh CLI から自動取得) | GitHub Token |
| `OPENAI_API_KEY` | ― | OpenAI API を使う場合に必要 |
| `LLM_LOCAL_URL` | `"http://localhost:11434/v1"` | ローカル LLM の API エンドポイント |